**import libraries**

In [19]:
import pandas as pd 


**load all dataframes**

In [20]:
df_2009_2010 = pd.read_parquet("../data/cleaned_2009_2010.parquet")
df_2010_2011 = pd.read_parquet("../data/cleaned_2010_2011.parquet")

**combined by concatination**

In [21]:
overlap = (df_2009_2010["year"] == 2010) & (df_2009_2010["month"] == "December")

df_all = pd.concat(
    [df_2009_2010[~overlap], df_2010_2011],
    ignore_index=True
)

In [22]:
df_all.head()

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,year,month,day,hour,returned,gross_sales,net_sales
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085.0,united kingdom,2009,December,Tuesday,7,no,83.4,83.4
1,489434,79323p,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,2009,December,Tuesday,7,no,81.0,81.0
2,489434,79323w,white cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,2009,December,Tuesday,7,no,81.0,81.0
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085.0,united kingdom,2009,December,Tuesday,7,no,100.8,100.8
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085.0,united kingdom,2009,December,Tuesday,7,no,30.0,30.0


In [23]:
# December 2010 should now appear once, from the second file only
dec_2010 = df_all[(df_all["year"] == 2010) & (df_all["month"] == "December")]
print(dec_2010["net_sales"].sum())

# compare with the second file alone, these should match
print(df_2010_2011[(df_2010_2011["year"] == 2010) & (df_2010_2011["month"] == "December")]["net_sales"].sum())

748957.02
748957.02


In [24]:
df_all["description"].unique().tolist()

['15cm christmas glass ball 20 lights',
 'pink cherry lights',
 'white cherry lights',
 'record frame 7" single size',
 'strawberry ceramic trinket box',
 'pink doughnut trinket pot',
 'save the planet mug',
 'fancy font home sweet home doormat',
 'cat bowl',
 'dog bowl , chasing ball design',
 'heart measuring spoons large',
 'lunchbox with cutlery fairy cakes',
 'door mat black flock',
 'love building block word',
 'home building block word',
 'assorted colour bird ornament',
 'peace wooden block letters',
 'christmas craft white fairy',
 'heart ivory trellis large',
 'heart filigree dove large',
 'full english breakfast plate',
 'pizza plate in box',
 'black diner wall clock',
 'set of 3 black flying ducks',
 'area patrolled metal sign',
 'please one person  metal sign',
 'bath building block word',
 'classic white frame',
 'small marshmallows pink bowl',
 'biscuits small bowl light blue',
 'scottie dog hot water bottle',
 'christmas craft heart decorations',
 'christmas craft heart

In [30]:
import numpy as np

# ORDER MATTERS: np.select uses the first rule that matches
RULES = [
    ("non-product / adjustments",
     r"^(?:postage|discount|short)$|\bmixed$"),

    ("christmas & seasonal decorations",
     r"christmas|xmas|wreath|baubles|santa|reindeer|advent|easter|stocking|gingerbread|sleigh|icicle|snowflake|\bnoel\b|holly|tinsel|tree decoration|angel decoration|folkart.*dec|scandinavian heart dec|woolly hat sock glove"),

    # signs/letters/doorstops early so 'bath', 'chocolate', 'party' in a sign name don't misfile it
    ("home decor & ornaments",
     r"\bsign\b|door hanger|door ?sign|doorstop|blackboard|block word|block letters|wood letters"),

    ("toys & games",
     r"(?<!\bbag )(?<!\blola )\btoys?\b|(?<!russian )\bdolls?\b|teddy|\bgames?\b|puzzle|jigsaw|snakes & ladders|bingo|domino|skittles|skipping|marbles|racing car|helicopter|gliders|rattle|\bmobile\b|monster|eight piece|space frog|treasure chest|snap cards|kit cards|windmill|crocheted animal|crochet (?:bear|cat)|larry the lamb|baby mouse|lamb with pants|stripey mice|magic (?:sheep|garden)"),

    ("party & gift supplies",
     r"\bparty\b|balloons?(?! writing)|bunting|\bwraps?\b|gift (?:wraps?|bags?|tags?)|ribbons?|\bcards?\b(?! holder)|napkins?(?! rings)|paper (?:cups|plates|chain)|doilies|straws|cake cases|mini cases|tissues|tissue ream|honeycomb|(?<!\bbath )confetti|invites"),

    ("stationery & office supplies",
     r"notebook|journal|sketchbook|\bpens?\b|pencils?|bookmark|stationery|notepad|memo|pocket book|writing set|chalk|stickers?|tattoos|erasers?"),

    ("jewellery & hair accessories",
     r"necklac|neckl\.|bracelet|earring|brooch|expandable ring|bubble gum ring|key ?rings?|key fob|hair ?(?:slides?|comb|clips?|band)|\bclips\b|charm\b|jewellery|jewelry"),

    ("bathroom & personal care",
     r"soap|toothbrush|\bbath\b|bathroom|sponge|toilet|wash ?bag|cosmetic bag|shower"),

    ("craft & sewing",
     r"\bkit\b|sewing|knitting|iron on patches|stamp set|feltcraft(?!.*\bbag\b)|glass gems"),

    ("candles, lighting & fragrance",
     r"candl|t-?light|tlight|lantern|lamp|lights?\b|torch|matches|incense|sachet|potpourri|scent"),

    ("textiles & soft furnishings",
     r"cushion|curtain|blanket|\bthrow\b|towel|apron|scarf|c/cover|hot water bottle|hottie|warmer|slipper|door ?mat|flannel|filler pad|gloves?\b"),

    ("kitchenware & dining",
     r"\bmugs?\b|\bcups?\b|saucer|\bplates?\b|\bbowls?\b|cutlery|teapot|tea set|coffee set|breakfast|kitchen|cooking|baking|\bpan\b|frying|\bjugs?\b|butter dish|\bdish\b|toastrack|toast its|cake stand|cake tins?|chopsticks|measuring spoons|\bladle|salad servers|cookie cutters|food (?:cover|container)|lunch ?box|place setting|coasters?|placemats?|napkin rings|aperitif|wine (?:glass|goblet)|milkshake glass|tea glass|chalice|goblet|french enamel|popcorn|\btray\b|utensil|egg cup"),

    ("bags & travel accessories",
     r"(?<!\btea )(?<!\bin )(?<!\bcotton )(?<!\bgauze )bag|purse|wallet|luggage|passport|shopper|\btote\b|\bcases?\b|umbrella|parasol|rain mac|rain hat"),

    ("garden & outdoor",
     r"garden|planter|flower pot|birdhouse|bird feeder|secateurs|potting shed|seed|wasp|twine"),

    ("tools & electronics",
     r"tool set|tape measure|calculator|headphones?|speaker"),

    ("storage & household organization",
     r"storage|\bbox(?:es)?\b|basket|\bjar\b|\btins?\b|hook|trinket|chest|cabinet|\brack\b|shelf|organiser|card holder|\bpegs?\b|duster"),

    ("food & drink",
     r"chocolates?|\bcandy\b|\bsweets\b|marshmallow|\btea\b|coffee|biscuit|licorice|jelly"),

    # explicit home decor keywords (last, so every other rule gets first pick)
    ("home decor & ornaments",
     r"ornament|figurine|frame|wall art|decoration|\bdecs?\b|mirror|clock|garland|ducks|magnet|knob|hanging|heart|dove|bird|piggy bank|globe|disco ball|photo cube|p'weight|mushroom|butterfl|trellis|filigree|hand open shape|wire|zinc|toadstool|egg house|slate|bottle|crochet|felt|wood"),
]

# anything with a description that matches NO rule above
DEFAULT_CATEGORY = "other"

desc = df_all["description"].fillna("").str.lower().str.strip()

conditions = [desc.str.contains(pattern, regex=True) for _, pattern in RULES]
categories = [name for name, _ in RULES]

df_all["category"] = np.select(conditions, categories, default=DEFAULT_CATEGORY)

# only missing/blank descriptions are "unknown"
df_all.loc[desc.eq(""), "category"] = "unknown"

In [31]:
df_all.head()

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,year,month,day,hour,returned,gross_sales,net_sales,category
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085.0,united kingdom,2009,December,Tuesday,7,no,83.4,83.4,christmas & seasonal decorations
1,489434,79323p,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,2009,December,Tuesday,7,no,81.0,81.0,"candles, lighting & fragrance"
2,489434,79323w,white cherry lights,12,2009-12-01 07:45:00,6.75,13085.0,united kingdom,2009,December,Tuesday,7,no,81.0,81.0,"candles, lighting & fragrance"
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085.0,united kingdom,2009,December,Tuesday,7,no,100.8,100.8,home decor & ornaments
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085.0,united kingdom,2009,December,Tuesday,7,no,30.0,30.0,storage & household organization


In [32]:
df_all["category"].unique()

<ArrowStringArray>
['christmas & seasonal decorations',    'candles, lighting & fragrance',
           'home decor & ornaments', 'storage & household organization',
             'kitchenware & dining',      'textiles & soft furnishings',
                     'toys & games',            'party & gift supplies',
     'stationery & office supplies',        'bags & travel accessories',
                            'other',        'non-product / adjustments',
                   'craft & sewing',         'bathroom & personal care',
                 'garden & outdoor',     'jewellery & hair accessories',
              'tools & electronics',                          'unknown',
                     'food & drink']
Length: 19, dtype: str

In [40]:
check = df_all.iloc[df_all["category"] == "kitchenware & dining"]
check.head(25)

,invoice,stockcode,description,quantity,invoicedate,price,customerid,country,year,month,day,hour,returned,gross_sales,net_sales,category
6,489434,21871,save the planet mug,24,2009-12-01 07:45:00,1.25,13085.0,united kingdom,2009,December,Tuesday,7,no,30.00,30.00,kitchenware & dining
8,489435,22350,cat bowl,12,2009-12-01 07:46:00,2.55,13085.0,united kingdom,2009,December,Tuesday,7,no,30.60,30.60,kitchenware & dining
9,489435,22349,"dog bowl , chasing ball design",12,2009-12-01 07:46:00,3.75,13085.0,united kingdom,2009,December,Tuesday,7,no,45.00,45.00,kitchenware & dining
10,489435,22195,heart measuring spoons large,24,2009-12-01 07:46:00,1.65,13085.0,united kingdom,2009,December,Tuesday,7,no,39.60,39.60,kitchenware & dining
11,489435,22353,lunchbox with cutlery fairy cakes,12,2009-12-01 07:46:00,2.55,13085.0,united kingdom,2009,December,Tuesday,7,no,30.60,30.60,kitchenware & dining
20,489436,22109,full english breakfast plate,16,2009-12-01 09:06:00,3.39,13078.0,united kingdom,2009,December,Tuesday,9,no,54.24,54.24,kitchenware & dining
21,489436,22107,pizza plate in box,4,2009-12-01 09:06:00,3.75,13078.0,united kingdom,2009,December,Tuesday,9,no,15.00,15.00,kitchenware & dining
28,489436,84596f,small marshmallows pink bowl,8,2009-12-01 09:06:00,1.25,13078.0,united kingdom,2009,December,Tuesday,9,no,10.00,10.00,kitchenware & dining
40,489437,37370,retro coffee mugs assorted,12,2009-12-01 09:08:00,1.25,15362.0,united kingdom,2009,December,Tuesday,9,no,15.00,15.00,kitchenware & dining
65,489438,84519a,tomato charlie+lola coaster set,56,2009-12-01 09:24:00,2.15,18102.0,united kingdom,2009,December,Tuesday,9,no,120.40,120.40,kitchenware & dining


In [35]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 16 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1044848 non-null  string        
 1   stockcode    1044848 non-null  string        
 2   description  1040573 non-null  string        
 3   quantity     1044848 non-null  int64         
 4   invoicedate  1044848 non-null  datetime64[us]
 5   price        1044848 non-null  float64       
 6   customerid   809561 non-null   float64       
 7   country      1044848 non-null  str           
 8   year         1044848 non-null  int32         
 9   month        1044848 non-null  str           
 10  day          1044848 non-null  str           
 11  hour         1044848 non-null  int32         
 12  returned     1044848 non-null  str           
 13  gross_sales  1044848 non-null  float64       
 14  net_sales    1044848 non-null  float64       
 15  category     1044848 non-n

**save as parquet**

In [36]:
df_all.to_parquet("../data/combined_all.parquet")